In [13]:
import pandas as pd
import glob
import json
import os

sample_path = f'{os.getcwd()}/../workspace/samples/'

gemini_pro_samples = 'x86-gemini-pro'
gemini_flash_samples = 'x86-gemini-flash'

def get_rows(path):

    summaries = sorted(glob.glob(f'{path}/*/agent_repair/summary.json'))
    experiment_results_path = f'{path}/results.json'

    with open(experiment_results_path, 'r') as f:
        experiment_results = json.load(f)
    
    sample_results = {
        sample['sample']: sample for sample in experiment_results['samples']
    }

    rows = []
    for sample, path in enumerate(summaries):

        with open(path, 'r') as f:
            data = json.load(f)
        
        arch = data['summary']['arch']
        model = data['models']['llm']

        total_input_tokens = data['llm_token_usage']['input_tokens']
        total_output_tokens = data['llm_token_usage']['output_tokens']
        total_total_tokens = data['llm_token_usage']['total_tokens']

        for attempt in data['attempts']:
            rows.append({
            'sample': sample,
            'arch': arch,
            'model': model,
            'attempt': attempt['attempt'],
            'boot_succeeded': attempt['summary']['boot_succeeded'],
            'define': attempt['changes']['define'],
            'undefine': attempt['changes']['undefine'],
            'original_config': f'{os.path.dirname(path)}/../.config',
            'modified_config': f'{os.path.dirname(path)}/attempt_{attempt["attempt"]}/modified.config',
            'edit_distance': data['summary']['edit_distance'],
            'total_constraints': data['summary']['total_constraints'],
            'total_input_tokens': total_input_tokens,
            'total_output_tokens': total_output_tokens,
            'total_total_tokens': total_total_tokens,
            'define': attempt['changes']['define'],
            'undefine': attempt['changes']['undefine'],
            'duration': sample_results[sample + 1]['duration'] / 60 / 60,
        })

    return rows

rows = get_rows(f'{sample_path}/{gemini_pro_samples}') + get_rows(f'{sample_path}/{gemini_flash_samples}')
df = pd.DataFrame(rows)

flash_samples = df[df['model'] == 'gemini-3.5-flash']

print(df.groupby('model')['sample'].nunique())

df.head()

model
gemini-3.1-pro-preview    50
gemini-3.5-flash          50
Name: sample, dtype: int64


,sample,arch,model,attempt,boot_succeeded,define,undefine,original_config,modified_config,edit_distance,total_constraints,total_input_tokens,total_output_tokens,total_total_tokens,duration
0,0,x86_64,gemini-3.1-pro-preview,0,no,None,None,/Users/ryangarfinkel/Documents/UCF/Thesis/KBoo...,/Users/ryangarfinkel/Documents/UCF/Thesis/KBoo...,2629,79,3061379,102456,3163835,4.953294
1,0,x86_64,gemini-3.1-pro-preview,1,timeout,[],[CONFIG_KASAN],/Users/ryangarfinkel/Documents/UCF/Thesis/KBoo...,/Users/ryangarfinkel/Documents/UCF/Thesis/KBoo...,2629,79,3061379,102456,3163835,4.953294
2,0,x86_64,gemini-3.1-pro-preview,2,panic,"[CONFIG_TTY, CONFIG_SERIAL_8250, CONFIG_SERIAL...",[CONFIG_KASAN],/Users/ryangarfinkel/Documents/UCF/Thesis/KBoo...,/Users/ryangarfinkel/Documents/UCF/Thesis/KBoo...,2629,79,3061379,102456,3163835,4.953294
3,0,x86_64,gemini-3.1-pro-preview,3,timeout,[],"[CONFIG_KASAN, CONFIG_KCSAN]",/Users/ryangarfinkel/Documents/UCF/Thesis/KBoo...,/Users/ryangarfinkel/Documents/UCF/Thesis/KBoo...,2629,79,3061379,102456,3163835,4.953294
4,0,x86_64,gemini-3.1-pro-preview,4,panic,"[CONFIG_TTY, CONFIG_SERIAL_8250, CONFIG_SERIAL...","[CONFIG_KASAN, CONFIG_KCSAN]",/Users/ryangarfinkel/Documents/UCF/Thesis/KBoo...,/Users/ryangarfinkel/Documents/UCF/Thesis/KBoo...,2629,79,3061379,102456,3163835,4.953294


In [14]:
def parse_config(path):
    with open(path, 'r') as f:
        lines = f.readlines()

    options = set()

    for line in lines:
        if '=' in line:
            options.add(line.strip().split('=')[0])
            
    return options

def jaccard_similarity(a, b):
    return len(a & b) / len(a | b) if a | b else 1.0

In [15]:
# Attempts

attempt_distribution = df.groupby(['model', 'sample']).max('attempt').reset_index()

print(attempt_distribution.groupby('model')['attempt'].describe().to_string())

                        count   mean       std  min  25%   50%   75%   max
model                                                                     
gemini-3.1-pro-preview   50.0   9.74  5.058071  2.0  6.0   8.5  11.0  20.0
gemini-3.5-flash         50.0  11.98  5.716035  4.0  7.0  10.5  18.5  20.0


In [16]:
# Token Usage

cols = ['total_input_tokens', 'total_output_tokens', 'total_total_tokens']

print(attempt_distribution.groupby('model')[cols].describe().to_string())

                       total_input_tokens                                                                                   total_output_tokens                                                                           total_total_tokens                                                                                
                                    count        mean           std       min         25%        50%         75%        max               count       mean           std      min       25%      50%        75%       max              count        mean           std       min        25%        50%        75%        max
model                                                                                                                                                                                                                                                                                                                       
gemini-3.1-pro-preview               50.0  192908

In [17]:
# Duration

print(attempt_distribution.groupby('model')['duration'].describe().to_string())

                        count      mean      std       min       25%       50%       75%        max
model                                                                                              
gemini-3.1-pro-preview   50.0  4.371065  2.82939  0.903242  2.317728  3.627117  4.943106  13.689475
gemini-3.5-flash         50.0  4.477639  2.65549  1.022747  2.494984  3.948610  6.845741  10.483128


In [24]:
# Sample Preservation

repaired_samples = df[df['boot_succeeded'] == 'yes'].copy()

repaired_samples['before_after_similarity'] = repaired_samples.apply(
    lambda row: jaccard_similarity(
        parse_config(row['original_config']),
        parse_config(row['modified_config'])
    ),
    axis=1
)

repaired_samples['num_defines'] = repaired_samples['define'].apply(len)
repaired_samples['num_undefines'] = repaired_samples['undefine'].apply(len)
repaired_samples['num_changes'] = repaired_samples['num_defines'] + repaired_samples['num_undefines']

print(repaired_samples.groupby('model')['before_after_similarity'].describe().to_string())

cols = ['num_defines', 'num_undefines', 'num_changes', 'edit_distance']
print(repaired_samples.groupby('model')[cols].describe().to_string())

                        count      mean       std       min       25%       50%       75%       max
model                                                                                              
gemini-3.1-pro-preview   45.0  0.851343  0.088025  0.613361  0.826946  0.852813  0.913010  0.986170
gemini-3.5-flash         39.0  0.870199  0.087613  0.572953  0.830690  0.884780  0.939937  0.986433
                       num_defines                                                    num_undefines                                                    num_changes                                                      edit_distance                                                                 
                             count       mean        std  min   25%   50%   75%   max         count       mean        std  min   25%   50%   75%   max       count       mean        std   min   25%   50%   75%    max         count         mean          std    min     25%     50%     75%     max
model     

In [23]:
from scipy.stats import ttest_ind

repaired_pro_samples = repaired_samples[repaired_samples['model'] == 'gemini-3.1-pro-preview']
repaired_flash_samples = repaired_samples[repaired_samples['model'] == 'gemini-3.5-flash']

t_stat, p_value = ttest_ind(repaired_pro_samples['before_after_similarity'], repaired_flash_samples['before_after_similarity'])

print(f't-statistic: {t_stat}')
print(f'p-value: {p_value}')


t-statistic: -0.9812968339353646
p-value: 0.3293328194444372


In [25]:
# Sample similarity

import numpy as np

n = len(repaired_flash_samples)

before_scores = np.zeros((n, n))
after_scores = np.zeros((n, n))

for i, a in enumerate(repaired_flash_samples['sample'].unique()):
    for j, b in enumerate(repaired_flash_samples['sample'].unique()):

        if i > j:
            continue

        config_a_before = parse_config(repaired_flash_samples[repaired_flash_samples['sample'] == a]['original_config'].iloc[0])
        config_b_before = parse_config(repaired_flash_samples[repaired_flash_samples['sample'] == b]['original_config'].iloc[0])

        before_scores[i, j] = jaccard_similarity(config_a_before, config_b_before)
        before_scores[j, i] = before_scores[i, j]

        config_a_after = parse_config(repaired_flash_samples[repaired_flash_samples['sample'] == a]['modified_config'].iloc[0])
        config_b_after = parse_config(repaired_flash_samples[repaired_flash_samples['sample'] == b]['modified_config'].iloc[0])
        
        after_scores[i, j] = jaccard_similarity(config_a_after, config_b_after)
        after_scores[j, i] = after_scores[i, j]

sample_similarity = pd.DataFrame({
    'before': before_scores[np.triu_indices_from(before_scores, k=1)],
    'after': after_scores[np.triu_indices_from(after_scores, k=1)],
})

print(sample_similarity.describe().to_string())

           before       after
count  741.000000  741.000000
mean     0.308385    0.325582
std      0.039162    0.037415
min      0.210565    0.236648
25%      0.280433    0.298733
50%      0.305782    0.322591
75%      0.332046    0.351470
max      0.438393    0.441643


In [26]:
from statsmodels.stats.contingency_tables import mcnemar

df['succeeded'] = df['boot_succeeded'] == 'yes'

last_attempts = df.sort_values('attempt').groupby(['sample', 'model']).last().reset_index()

pivot = last_attempts.pivot_table(
    index='sample',
    columns='model',
    values='succeeded',
)
table = pd.crosstab(pivot['gemini-3.1-pro-preview'], pivot['gemini-3.5-flash'])
result = mcnemar(table, exact=True)

print(f'McNemar test statistic: {result.statistic}, p-value: {result.pvalue}')

print('Contingency Table:')
print(table.to_string())

McNemar test statistic: 2.0, p-value: 0.109375
Contingency Table:
gemini-3.5-flash        0.0  1.0
gemini-3.1-pro-preview          
0.0                       3    2
1.0                       8   37
